# Train and Predict Drones

### Abstract

...


### Introduction

...


### Dataset



# 🏁 1. Initialization

pip install `ultralytics` and [dependencies](https://github.com/ultralytics/ultralytics/blob/main/pyproject.toml) and check software and hardware.

[![PyPI - Version](https://img.shields.io/pypi/v/ultralytics?logo=pypi&logoColor=white)](https://pypi.org/project/ultralytics/) [![PyPI - Python Version](https://img.shields.io/pypi/pyversions/ultralytics?logo=python&logoColor=gold)](https://pypi.org/project/ultralytics/)

### 1.0 Installing dependencies

In [ ]:
!pip install ultralytics markdown rich wrapt pandas huggingface_hub scikit-learn opencv-python wandb python-dotenv datasets -q

### 1.1 Importing Libraries

In [ ]:
from datasets import load_dataset, Image, concatenate_datasets, DatasetDict
from IPython.display import display, Image as IPyImage
from ultralytics import YOLO, settings
from typing import Iterable, Union
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from datetime import datetime
from pathlib import Path
from PIL import Image

import ultralytics
import datetime
import shutil
import wandb
import uuid
import yaml
import math
import re
import os

ultralytics.checks()

### 1.2 Global Definitions

In [ ]:
DATASET_ROOT_DIR = Path('./datasets/main')
DATASET_ALL_DIR = DATASET_ROOT_DIR / 'train_validation_test'
TRAINING_DATASET_DIRECTORY = DATASET_ROOT_DIR / 'train'
VALIDATION_DATASET_DIRECTORY = DATASET_ROOT_DIR / 'valid/'
TEST_DATASET_DIRECTORY = DATASET_ROOT_DIR / 'test/'

IMAGE_APPLY_GRAYSCALE = True

MODELS_DIRECTORY = Path('./models/')
load_dotenv()

### 1.3 Global Settings

In [ ]:
# YOLO settings
settings.update({"wandb": True})

# Initialize Weights & Biases environment
wandb.login(key=os.getenv("WANDB_TOKEN"))

# login("TOKEN") # Keep commented if token loaded from .env file

### 1.4 Global Structure

In [ ]:
dataset_structure = {
    "root": Path(""),
    "name": "",
    "classes": [],
    "train": {
        "images": [],
        "labels": [],
    },
    "valid": {
        "images": [],
        "labels": [],
    },
    "test": {
        "images": [],
        "labels": [],
    },
}


### 1.5 Global Function Definitions

In [ ]:
def list_files(
        directory: Union[str, Path],
        extensions: Iterable[str],
        include_root_directory: bool = False,
        recursive: bool = False,
) -> list[Path]:
    directory = Path(directory)
    extensions = tuple(extensions)

    matched_files = []

    if recursive:
        iterator = directory.rglob("*")
    else:
        iterator = directory.iterdir()

    for p in iterator:
        if p.is_file() and p.suffix in extensions:
            matched_files.append(
                p if include_root_directory else p.name
            )

    return matched_files


def numeric_key(name):
    """Extract the first number from a filename for sorting."""
    nums = re.findall(r'\d+', name)
    return int(nums[0]) if nums else float('inf')


def sort_files_by_number(files_to_sort: list):
    """
    Sort a list of filenames by the first number found in each name.

    Args:
        files_to_sort (list): List of filenames (strings)

    Returns:
        list: Sorted list of filenames
    """
    return sorted(files_to_sort, key=lambda i: int(i.stem))


def update_dataset_structure():
    dataset_structure["root"] = Path(DATASET_ROOT_DIR)
    dataset_structure["name"] = DATASET_ROOT_DIR.name

    dataset_structure["train"]["images"] = sort_files_by_number(
        list_files(TRAINING_DATASET_DIRECTORY, [".jpg", ".jpeg", ".JPG", ".JPEG"], True))
    dataset_structure["train"]["labels"] = sort_files_by_number(list_files(TRAINING_DATASET_DIRECTORY, [".txt"], True))

    dataset_structure["valid"]["images"] = sort_files_by_number(
        list_files(VALIDATION_DATASET_DIRECTORY, [".jpg", ".jpeg", ".JPG", ".JPEG"], True))
    dataset_structure["valid"]["labels"] = sort_files_by_number(
        list_files(VALIDATION_DATASET_DIRECTORY, [".txt"], True))

    dataset_structure["test"]["images"] = sort_files_by_number(
        list_files(VALIDATION_DATASET_DIRECTORY, [".jpg", ".jpeg", ".JPG", ".JPEG"], True))
    dataset_structure["test"]["labels"] = sort_files_by_number(list_files(VALIDATION_DATASET_DIRECTORY, [".txt"], True))

    dataset_structure["classes"] = ["drone", "other"]


def delete_files_in_dataset(files_to_delete: list):
    try:
        confirm = input("Files are going to be deleted. Type 'yes' to continue: ").strip().lower()
        if confirm != 'yes':
            print("Deletion aborted by user.")
            return

        for file in files_to_delete:
            if os.path.isfile(file):
                os.remove(file)
                print(f"Deleted: {file}")
            else:
                print(f"Warning: File does not exist: {file}")

    except KeyboardInterrupt:
        print("\nDeletion aborted by user (KeyboardInterrupt).")
    finally:
        try:
            update_dataset_structure()
        except NameError:
            pass


def backup_dataset():
    dataset_path = dataset_structure.get("path", "")
    backup_dir = os.path.join(dataset_path, "backup")
    os.makedirs(backup_dir, exist_ok=True)

    now = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
    target_name = f"{dataset_structure["name"]}-{now}"
    target_path = os.path.join(backup_dir, target_name)

    def ignore_backup(_, names):
        return {"backup"} if "backup" in names else set()

    shutil.copytree(dataset_path, target_path, ignore=ignore_backup)
    print(f"Backup created at: {target_path}")
    return str(target_path)


def interpret_map(map_value: float) -> str:
    """
    Interpret mAP value according to standard object detection heuristics.
    """
    if map_value < 0.10:
        return "Model is effectively failing"
    elif map_value < 0.30:
        return "Very weak performance"
    elif map_value < 0.50:
        return "Usable baseline"
    elif map_value < 0.70:
        return "Good performance"
    else:
        return "Strong performance"

def plot_image_grid(images_path, nb_cols = 4, max_images_preview = -1, show_title = False):
    if max_images_preview != -1:
        images_path = images_path[:max_images_preview]

    rows = math.ceil(len(images_path) / nb_cols)

    img = Image.open(images_path[0])
    w, h = img.size  # pixels

    dpi = 100

    img_w = w / dpi
    img_h = h / dpi
    #
    plt.figure(figsize=(nb_cols * img_w, rows * img_h))

    for i, path in enumerate(images_path):
        img = Image.open(path)
        plt.subplot(rows, nb_cols, i + 1)
        plt.imshow(img)
        plt.axis("off")
        if show_title:
            plt.title(path.name, fontsize=25)

    plt.tight_layout()
    plt.show()

# 📂 2. Dataset

### 2.0 Acquire Dataset

Download dataset from hugging face.

In [ ]:
dataset = load_dataset("Hibou-Foundation/computer-vision")

### 2.1 Split dataset

Split the dataset into train, validation, test.

In [ ]:
base_split = "train_validation_test"
label_column = "class_id"

train_ratio = [0.8, 0.8]  # [class 0, class 1]
valid_ratio = [0.1, 0.1]
test_ratio = [0.1, 0.1]

seed = 42

for i in range(len(train_ratio)):
    assert train_ratio[i] + valid_ratio[i] + test_ratio[i] == 1.0

train_parts = []
valid_parts = []
test_parts = []

num_classes = len(train_ratio)

for cls in range(num_classes):
    cls_ds = dataset[base_split].filter(
        lambda x: x[label_column] == cls
    )

    cls_ds = cls_ds.shuffle(seed=seed)

    n = len(cls_ds)
    n_train = int(n * train_ratio[cls])
    n_valid = int(n * valid_ratio[cls])

    train_parts.append(cls_ds.select(range(0, n_train)))
    valid_parts.append(cls_ds.select(range(n_train, n_train + n_valid)))
    test_parts.append(cls_ds.select(range(n_train + n_valid, n)))

train_ds = concatenate_datasets(train_parts).shuffle(seed=seed)
valid_ds = concatenate_datasets(valid_parts).shuffle(seed=seed)
test_ds = concatenate_datasets(test_parts).shuffle(seed=seed)

dataset = DatasetDict({
    "train": train_ds,
    "validation": valid_ds,
    "test": test_ds,
})
dataset

### 2.2 Convert to file images

In [ ]:
# Create folders
for split in ["train", "valid", "test"]:
    os.makedirs(f"{DATASET_ROOT_DIR}/{split}", exist_ok=True)


def export_to_yolo(ds, split_name):
    for idx, sample in enumerate(ds):
        image = sample["image"]  # already a PIL.Image
        label = sample["raw_label"]  # already YOLO format [[class, cx, cy, w, h], ...]
        img_name = sample["name"]
        txt_name = img_name.split(".")[0] + ".txt"

        # Save image
        img_path = f"{DATASET_ROOT_DIR}/{split_name}/{img_name}"
        image.save(img_path, quality=95)

        # Save labels
        lbl_path = f"{DATASET_ROOT_DIR}/{split_name}/{txt_name}"
        with open(lbl_path, "w") as f:
            f.write(label)


# Run export
split_mapping = {"train": "train", "validation": "valid", "test": "test"}
for hf_split, folder_name in split_mapping.items():
    export_to_yolo(dataset[hf_split], folder_name)
update_dataset_structure()

# 📚️ 3. Models Settings

### 3.1 Model selection

In [ ]:
selected_size = "nano"
selected_version = "26"

YOLO_MODEL_SIZE = {
    "nano": "n",
    "small": "s",
    "medium": "m",
    "large": "l",
    "xlarge": "x",
}
run_session_id = str(uuid.uuid4()).split("-")[0]  # For wandb
model_name = f"yolo{selected_version}{YOLO_MODEL_SIZE[selected_size]}.pt"
model_path = MODELS_DIRECTORY / model_name
model = YOLO(model_path, task="detect")
print(f"Session ID: {run_session_id}")

### 3.2 Training Configuration
Define hyperparameters (epochs, batch size, image size)

In [ ]:
train_config = {
    'epochs': 170,
    'imgsz': 640,
    'batch': 16,
    'lr0': 0.001,
    'patience': 20,
    'optimizer': 'adamW',

    # Augmentation Settings
    'degrees': 20,
    'perspective': 0.002,
    'mosaic': 0.6,
    'close_mosaic': 50,
    'cutmix': 0.2,

    'project': 'computer-vision',
    'device': [0, 1]
}
train_config

**Parameter breakdown:**

- **train**: Executes the YOLOv11x training pipeline.
- **model**=yolov11x.pt: Uses pre-trained YOLOv11x weights as initialization.
- **data**=/content/data.yaml: Specifies the dataset configuration file.
- **imgsz**=640: Sets input resolution to enhance small-object detection.
- **lr0**=0.001: Sets the learning rate for each training step.
- **epochs**=32: Defines the number of training cycles over the dataset.
- **batch**=16: Sets the batch size for each training step.
- **device**=0: Allocates GPU device 0 for training.
- **optimizer**=AdamW: As default, AdamW optimization algorithm utilized.

# ⚙️ 5. Training

Purpose: Handle model setup and training configuration.

### 5.1 Save dataset configuration

In [ ]:
data_yaml = dict(
    train=os.path.join('../../', TRAINING_DATASET_DIRECTORY),
    val=os.path.join('../../', VALIDATION_DATASET_DIRECTORY),
    nc=2,
    channels = 1 if IMAGE_APPLY_GRAYSCALE else 3,
    names=['drone', 'other']
)

data_config_path = DATASET_ROOT_DIR / 'data.yaml'

with open(data_config_path, 'w') as outfile:
    yaml.dump(data_yaml, outfile, default_flow_style=True)
%cat "$data_config_path"

### 5.2 Run Training

In [ ]:
run_name = f"{selected_version}-{selected_size}-{run_session_id}"
results = model.train(**train_config,
                      data=data_config_path,
                      name=run_name)

### 5.3 Training results

In [ ]:
result_dir = Path(results.save_dir)
display(IPyImage(filename=str(result_dir / "results.png")))

In [ ]:
%matplotlib inline

# Retrieve val_batch images
val_batch = []
i = 0
batch_file = result_dir / f"val_batch{i}_labels.jpg"
while batch_file.exists():
    val_batch.append(batch_file)
    val_batch.append(result_dir / f"val_batch{i}_pred.jpg")
    i += 1
    batch_file = result_dir / f"val_batch{i}_labels.jpg"

# Retrieve confusion matrix images
confusion_matrix_path = [
    result_dir / "confusion_matrix.png",
    result_dir / "confusion_matrix_normalized.png"
]

# Retrieve metric images
boxes_path = [
    result_dir / "BoxF1_curve.png",
    result_dir / "BoxP_curve.png",
    result_dir / "BoxPR_curve.png",
    result_dir / "BoxR_curve.png",
]

# Show images
plot_image_grid(val_batch, nb_cols=2, show_title=True)
plot_image_grid(confusion_matrix_path, nb_cols=2)
plot_image_grid(boxes_path, nb_cols=2)


# ✅️ 6. Validation

###  6.1 Evaluate Model

In [ ]:
metrics = model.val()

In [ ]:
print(metrics.box.map)
print(metrics.box.map50)
print(metrics.box.map75)
print(metrics.box.maps)

# 🏭️ 7. Inference

Purpose: Evaluate results qualitatively and quantitatively.


### 7.0 Load model

In [ ]:
best_path = MODELS_DIRECTORY / "yolo11n_drone.pt"
# best_path = os.path.join("./computer-vision/nano11", "weights/best.pt")
custom_model = YOLO(best_path)

### 7.1 Run Predictions (IMAGES)

In [ ]:
predictions = custom_model.predict(
    TEST_DATASET_DIRECTORY,
    save=True,
    project=Path('computer-vision', run_name),
    conf=0.25
)
predictions_output_dir = predictions[0].save_dir

### 7.2 Visualize Predictions

In [ ]:
predictions_paths = list_files(predictions_output_dir, [".jpg", ".png", ".jpeg"], True)

plot_image_grid(predictions_paths, max_images_preview=100)

### 7.3 Run Predictions (VIDEOS)

In [ ]:
result = custom_model.track(
    source="/run/media/adrien/852ab715-3aaa-4da8-a8de-9f9524a00686/Datasets/PTZ/Videos/4 (trimmed).mp4",
    conf=0.3,
    iou=0.5,
    show=False,
    imgsz=640,
    save=True,
    exist_ok=True
)

# ⛴️ 8. Export & Deployment

### Supported Formats

- [Source](https://docs.ultralytics.com/modes/export/#arguments)

| Format                                             | `format` Argument |
|----------------------------------------------------|-------------------|
| [PyTorch](https://pytorch.org/)                    | -                 |
| [TorchScript](../integrations/torchscript.md)      | `torchscript`     |
| [ONNX](../integrations/onnx.md)                    | `onnx`            |
| [OpenVINO](../integrations/openvino.md)            | `openvino`        |
| [TensorRT](../integrations/tensorrt.md)            | `engine`          |
| [CoreML](../integrations/coreml.md)                | `coreml`          |
| [TF SavedModel](../integrations/tf-savedmodel.md)  | `saved_model`     |
| [TF GraphDef](../integrations/tf-graphdef.md)      | `pb`              |
| [TF Lite](../integrations/tflite.md)               | `tflite`          |
| [TF Edge TPU](../integrations/edge-tpu.md)         | `edgetpu`         |
| [TF.js](../integrations/tfjs.md)                   | `tfjs`            |
| [PaddlePaddle](../integrations/paddlepaddle.md)    | `paddle`          |
| [MNN](../integrations/mnn.md)                      | `mnn`             |
| [NCNN](../integrations/ncnn.md)                    | `ncnn`            |
| [IMX500](../integrations/sony-imx500.md){{ tip3 }} | `imx`             |
| [RKNN](../integrations/rockchip-rknn.md)           | `rknn`            |
| [ExecuTorch](../integrations/executorch.md)        | `executorch`      |
| [Axelera](../integrations/axelera.md)              | `axelera`         |

In [ ]:
model.export(format="onnx")